In [1]:
import pandas as pd
import sqlite3
import json
import os

def find_file(possible_names):
    """Helper function to find a file even if the name varies slightly."""
    for name in possible_names:
        if os.path.exists(name):
            return name
    raise FileNotFoundError(f"Could not find any of these files: {possible_names}")

def build_analytics_warehouse():
    print("🚀 Starting Zenith ETL Pipeline...")
    
    # Connect to the NEW Analytics Warehouse (This creates the file if it doesn't exist)
    db_path = 'zenith_analytics_warehouse.db'
    engine = sqlite3.connect(db_path)
    
    # ==========================================
    # STEP 1: EXTRACT & LOAD RAW DATA INTO WAREHOUSE
    # ==========================================
    print("📦 Loading Raw Data into Warehouse...")

    # 1A. Customer JSON
    json_file = find_file(['customer_profiles_v2.json'])
    with open(json_file, 'r') as f:
        data = json.load(f)
    parsed_customers = []
    for key, val in data.items():
        parsed_customers.append({
            'CustomerID': key.replace('usr_profile_', ''),
            'macro_region': val.get('account_details', {}).get('geo_segmentation', {}).get('macro_region'),
            'utm_medium': val.get('acquisition_telemetry', {}).get('source', {}).get('utm_medium'),
            'tier': val.get('financial_metrics_v2', {}).get('status', {}).get('tier'),
            'return_rate_pct': val.get('financial_metrics_v2', {}).get('risk_factors', {}).get('return_rate_pct')
        })
    df_cust = pd.DataFrame(parsed_customers)
    df_cust.to_sql('raw_customers', engine, if_exists='replace', index=False)

    # 1B. Core Transactions (from old SQLite)
    old_db_file = find_file(['zenith_core_db.sqlite'])
    old_conn = sqlite3.connect(old_db_file)
    df_trans = pd.read_sql_query("SELECT * FROM transactions_master", old_conn)
    old_conn.close()
    df_trans.to_sql('raw_transactions', engine, if_exists='replace', index=False)

    # 1C. CSV Files (PnL, SKU, Marketing)
    pnl_file = find_file(['monthly_pnl_summary.csv'])
    df_pnl = pd.read_csv(pnl_file)
    df_pnl.to_sql('raw_pnl', engine, if_exists='replace', index=False)

    sku_file = find_file(['sku_summary.csv'])
    df_sku = pd.read_csv(sku_file)
    df_sku.to_sql('raw_sku_summary', engine, if_exists='replace', index=False)

    # Handling the tricky Marketing File name
    mkt_file = find_file([
        'marketing_spend_history.xlsx - Sheet1.csv', 
        'marketing_spend_history.csv', 
        'marketing_spend_history.xlsx'
    ])
    if mkt_file.endswith('.xlsx'):
        df_mkt = pd.read_excel(mkt_file)
    else:
        df_mkt = pd.read_csv(mkt_file)
    df_mkt.to_sql('raw_marketing_spend', engine, if_exists='replace', index=False)

    # ==========================================
    # STEP 2: TRANSFORM & CREATE SINGLE SOURCE OF TRUTH
    # ==========================================
    print("⚙️ Transforming Data (Merging...)")
    df_merged = pd.merge(df_trans, df_cust, on='CustomerID', how='left')
    
    # Save the merged master table to the database
    df_merged.to_sql('master_transactions_enriched', engine, if_exists='replace', index=False)

    # ==========================================
    # STEP 3: VERIFY THE WAREHOUSE
    # ==========================================
    print("\n✅ ETL Complete! Verifying database contents...\n")
    print("-" * 50)
    print(f"{'Table Name':<30} | {'Row Count':<10}")
    print("-" * 50)
    
    # Query sqlite_master to get all tables in our new database
    tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table';", engine)
    
    for table_name in tables['name']:
        count = pd.read_sql_query(f"SELECT COUNT(*) as cnt FROM {table_name}", engine).iloc[0]['cnt']
        print(f"{table_name:<30} | {count:<10}")
    print("-" * 50)

    engine.close()
    print(f"\n📁 Your new database is ready: {os.path.abspath(db_path)}")

# Run the pipeline
if __name__ == "__main__":
    build_analytics_warehouse()

In [2]:
pip install openpyxl


   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpy


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
